# Hoeffding Tree Stream Demo

This notebook implements the split-decision rule used by Very Fast Decision Trees. A streaming tree does not wait to see all data. It uses the Hoeffding bound to decide when the best observed split is probably better than the runner-up.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(29)

In [ ]:
def stream(n=5000):
    for _ in range(n):
        x0 = rng.normal()
        x1 = rng.normal()
        x2 = rng.normal()
        # The true root split is x0 > 0, with some noise and a weaker x1 interaction.
        logit = 1.6 * (x0 > 0) + 0.45 * (x1 > 0) - 0.15 + rng.normal(0, 0.6)
        y = int(logit > 0.75)
        yield np.array([x0, x1, x2]), y

def entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts[counts > 0] / total
    return -np.sum(p * np.log2(p))

def info_gain_from_sides(left_counts, right_counts):
    parent = left_counts + right_counts
    total = parent.sum()
    return entropy_from_counts(parent) - (left_counts.sum() / total) * entropy_from_counts(left_counts) - (right_counts.sum() / total) * entropy_from_counts(right_counts)

In [ ]:
class BinaryThresholdObserver:
    def __init__(self, n_features, threshold=0.0):
        self.threshold = threshold
        self.left = np.zeros((n_features, 2), dtype=int)
        self.right = np.zeros((n_features, 2), dtype=int)
        self.n = 0

    def update(self, x, y):
        self.n += 1
        for j, value in enumerate(x):
            if value <= self.threshold:
                self.left[j, y] += 1
            else:
                self.right[j, y] += 1

    def gains(self):
        return np.array([
            info_gain_from_sides(self.left[j], self.right[j])
            for j in range(self.left.shape[0])
        ])

def hoeffding_epsilon(r, n, delta):
    return np.sqrt((r * r * np.log(1 / delta)) / (2 * n))

In [ ]:
observer = BinaryThresholdObserver(n_features=3, threshold=0.0)
delta = 1e-5
r = 1.0  # entropy range for binary classification
records = []
split_at = None

for i, (x, y) in enumerate(stream(), start=1):
    observer.update(x, y)
    if i % 25 == 0:
        gains = observer.gains()
        order = np.argsort(gains)[::-1]
        best, second = order[0], order[1]
        gap = gains[best] - gains[second]
        eps = hoeffding_epsilon(r, observer.n, delta)
        records.append({
            "n": i,
            "best_feature": f"x{best}",
            "best_gain": gains[best],
            "second_gain": gains[second],
            "gap": gap,
            "epsilon": eps,
            "commit": gap > eps,
        })
        if split_at is None and gap > eps:
            split_at = i
            break

history = pd.DataFrame(records)
history.tail()

In [ ]:
print("First split decision after", split_at, "examples")
print(history.tail(1).T)

In [ ]:
plt.figure(figsize=(7, 4.2))
plt.plot(history["n"], history["gap"], label="best minus second-best gain")
plt.plot(history["n"], history["epsilon"], label="Hoeffding bound")
plt.axvline(split_at, color="tab:red", linestyle="--", label="split committed")
plt.xlabel("streamed examples")
plt.ylabel("criterion difference")
plt.title("VFDT-style split commitment")
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()

In [ ]:
observer.gains()

## Takeaway

A streaming tree commits to a split when the observed gap between the best and second-best split exceeds the uncertainty bound. This is a different algorithmic regime from batch CART: the split decision is statistical and incremental.